In [46]:
import numpy as np
import pandas as pd
import pandas_datareader as pdr
import matplotlib.pyplot as plt

def get_bollinger_bands(data, rate=20, sd=2):
    sma = data['close'].rolling(rate).mean()
    std = data['close'].rolling(rate).std()
    bollinger_up = pd.Series(sma + std * sd, name = 'bollinger_up_' + str(rate)) # Calculate top band
    bollinger_down = pd.Series(sma - std * sd, name ='bollinger_down_' + str(rate)) # Calculate bottom band
    data = data.join([bollinger_up,bollinger_down])
    return data

# symbol = 'AAPL'
# df = pdr.DataReader(symbol, 'yahoo', '2014-07-01', '2015-07-01')
# df.index = np.arange(df.shape[0])




def SMA(data, ndays=20): 
    SMA = pd.Series(data['close'].rolling(ndays).mean(), name = 'SMA_'+ str(ndays)) 
    data = data.join(SMA) 
    return data

def EWMA(data, ndays=20): 
    EMA = pd.Series(data['close'].ewm(span = ndays, min_periods = ndays - 1).mean(), 
                 name = 'EWMA_' + str(ndays)) 
    data = data.join(EMA) 
    return data

def rsi(data, periods = 20):
    
    close_delta = data['close'].diff()

    # Make two series: one for lower closes and one for higher closes
    up = close_delta.clip(lower=0)
    down = -1 * close_delta.clip(upper=0)
    
    ma_up = up.ewm(com = periods - 1, adjust=True, min_periods = periods).mean()
    ma_down = down.ewm(com = periods - 1, adjust=True, min_periods = periods).mean()

    rsi = ma_up / ma_down
    rsi = 100 - (100/(1 + rsi))
    RSI = pd.Series(rsi, name='RSI_' + str(periods))
    data = data.join(RSI)
    return data

def gain(x):
    return ((x > 0) * x).sum()


def loss(x):
    return ((x < 0) * x).sum()


# Calculate money flow index
def mfi(data, n=20):
    high = data['high']
    low = data['low']
    close = data['close']
    volume = data['volume']
    typical_price = (high + low + close)/3
    money_flow = typical_price * volume
    mf_sign = np.where(typical_price > typical_price.shift(1), 1, -1)
    signed_mf = money_flow * mf_sign
    mf_avg_gain = signed_mf.rolling(n).apply(gain, raw=True)
    mf_avg_loss = signed_mf.rolling(n).apply(loss, raw=True)
    mfi = (100 - (100 / (1 + (mf_avg_gain / abs(mf_avg_loss)))))
    MFI = pd.Series(mfi, name='MFI_' + str(n))
    data = data.join(MFI)
    return data

def percent_change(data, level = 'close'):
    PER = pd.Series((data[level] - data[level].shift(1))/data[level].shift(1)*100, name = 'PER_'+level)
    data = data.join(PER)
    return data
    

In [56]:
import yfinance


symbols = pd.read_csv('nasdaq100.csv')['Symbol']
tttable = []
for symbol in symbols:
    print(symbol)
#     continue
    hist = yfinance.download(tickers=symbol,period="5d",interval="1m")
    
    if hist.empty:
        continue

    # change column name to lower case and without space.
    # option 1
    new_col_list = []
    for i in hist.columns:
        new_col_list.append(i.lower().replace(" ", ""))
    hist.columns = new_col_list

    # option 2 
    # dataframe.columns.values == array
    # dataframe.columns == index
    # pandas does not want pd.Indexs to be mutable
    # for i,j in enumerate(hist.columns):
    #     hist.columns.values[i] = j.lower().replace(" ", "")


    # in case zeros value in dataset, change zeros to ones for a better calculation
    hist.replace(0,1,inplace=True)

    length = range(7,28,7)
    st = range(1,4,1)

    final_table = []
    for ii in length:
        for jj in st:
            sub = hist.copy()
        #     hist = mfi(hist,14)

        #     hist = rsi(hist, 14)

            sub = get_bollinger_bands(sub, ii,jj)

        #     hist = percent_change(hist, 'close')

        #     hist = percent_change(hist, 'volume')

            sub['under_BBL'] = np.where(sub['close'] < sub['bollinger_down_'+str(ii)], 1, 0)

            tb = [0,0]
            for i in np.unique(sub.index.date):
    #             print(i,ii,jj)
                daily_hist = sub[str(i):str(i)]

    #             plt.figure(figsize=(18,8))
    #             plt.title(symbol + ' Bollinger Bands')
    #             plt.xlabel('Days')
    #             plt.ylabel('Closing Prices')
    #             plt.plot(daily_hist['close'], label='Closing Prices')
    #             plt.plot(daily_hist['bollinger_up_'+str(ii)], label='Bollinger Up', c='g')
    #             plt.plot(daily_hist['bollinger_down_'+str(ii)], label='Bollinger Down', c='r')




                for i in range(5):
                    daily_hist['high_'+str(i)] = daily_hist['high'].shift(-1-i)
                daily_hist['highest'] = daily_hist[["high_0", "high_1", "high_2", "high_3", "high_4"]].max(axis=1)
                daily_hist['success'] = np.where((daily_hist['under_BBL'] == 1) & (daily_hist['close']*1.001 < daily_hist['highest']), 1,0)

    #             try:
    #                 print(len(daily_hist[daily_hist['under_BBL']==1]), len(daily_hist[daily_hist['success']==1]), len(daily_hist[daily_hist['success']==1])/len(daily_hist[daily_hist['under_BBL']==1]))
    #             except ZeroDivisionError:
    #                 print(len(daily_hist[daily_hist['under_BBL']==1]), len(daily_hist[daily_hist['success']==1]), 0)

    #             plt.scatter(daily_hist[daily_hist['success']==1]['close'].index, daily_hist[daily_hist['success']==1]['close'],label='success', marker = '^', color = 'r')
    #             plt.scatter(daily_hist[(daily_hist['success']==0) & (daily_hist['under_BBL']==1)]['close'].index, daily_hist[(daily_hist['success']==0) & (daily_hist['under_BBL']==1)]['close'],label='fail', marker = 'v', color = 'b')
    #             plt.legend()
    #             plt.show()



                tb[0] +=len(daily_hist[daily_hist['under_BBL']==1])
                tb[1] +=len(daily_hist[daily_hist['success']==1])
            final_table.append([symbol,ii,jj,tb[0],tb[1]])
            tttable.append([symbol,ii,jj,tb[0],tb[1]])

    ttable = pd.DataFrame(final_table, columns = ['ticket','length','std','total','success'])

    ttable['s_rate'] = ttable['success']/ttable['total']

    ttable['expectation'] = 1.001**ttable['success']*0.999**(ttable['total']-ttable['success'])

#     print(ttable[ttable['s_rate']==ttable['s_rate'].max()])
#     print(ttable[ttable['expectation']==ttable['expectation'].max()])

#     ttable.to_csv('Results\\'+symbol+'.csv',index=False)

ttttable = pd.DataFrame(tttable, columns = ['ticket','length','std','total','success'])

ttttable['s_rate'] = ttttable['success']/ttttable['total']

ttttable['expectation'] = 1.001**ttttable['success']*0.999**(ttttable['total']-ttttable['success'])
ttttable.to_csv('Results\\''all.csv',index=False)

AAIT
[*********************100%***********************]  1 of 1 completed

1 Failed download:
- AAIT: No data found for this date range, symbol may be delisted
AAL
[*********************100%***********************]  1 of 1 completed


<ipython-input-56-aecd12de356d>:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  daily_hist['high_'+str(i)] = daily_hist['high'].shift(-1-i)
<ipython-input-56-aecd12de356d>:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  daily_hist['highest'] = daily_hist[["high_0", "high_1", "high_2", "high_3", "high_4"]].max(axis=1)
<ipython-input-56-aecd12de356d>:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See 

AAME
[*********************100%***********************]  1 of 1 completed


<ipython-input-56-aecd12de356d>:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  daily_hist['high_'+str(i)] = daily_hist['high'].shift(-1-i)
<ipython-input-56-aecd12de356d>:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  daily_hist['highest'] = daily_hist[["high_0", "high_1", "high_2", "high_3", "high_4"]].max(axis=1)
<ipython-input-56-aecd12de356d>:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See 

AAOI
[*********************100%***********************]  1 of 1 completed


...

<ipython-input-56-aecd12de356d>:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  daily_hist['high_'+str(i)] = daily_hist['high'].shift(-1-i)
<ipython-input-56-aecd12de356d>:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  daily_hist['highest'] = daily_hist[["high_0", "high_1", "high_2", "high_3", "high_4"]].max(axis=1)
<ipython-input-56-aecd12de356d>:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See 

ZFGN
[*********************100%***********************]  1 of 1 completed

1 Failed download:
- ZFGN: No data found, symbol may be delisted
ZGNX
[*********************100%***********************]  1 of 1 completed

1 Failed download:
- ZGNX: No data found, symbol may be delisted
ZHNE
[*********************100%***********************]  1 of 1 completed

1 Failed download:
- ZHNE: Period '5d' is invalid, must be one of ['1mo', '3mo', '6mo', 'ytd', '1y', '2y', '5y', '10y', 'max']
ZINC
[*********************100%***********************]  1 of 1 completed

1 Failed download:
- ZINC: Period '5d' is invalid, must be one of ['1mo', '3mo', '6mo', 'ytd', '1y', '2y', '5y', '10y', 'max']
ZION
[*********************100%***********************]  1 of 1 completed


<ipython-input-56-aecd12de356d>:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  daily_hist['high_'+str(i)] = daily_hist['high'].shift(-1-i)
<ipython-input-56-aecd12de356d>:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  daily_hist['highest'] = daily_hist[["high_0", "high_1", "high_2", "high_3", "high_4"]].max(axis=1)
<ipython-input-56-aecd12de356d>:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See 

ZIONW
[*********************100%***********************]  1 of 1 completed

1 Failed download:
- ZIONW: No data found, symbol may be delisted
ZIONZ
[*********************100%***********************]  1 of 1 completed

1 Failed download:
- ZIONZ: No data found, symbol may be delisted
ZIOP
[*********************100%***********************]  1 of 1 completed

1 Failed download:
- ZIOP: No data found, symbol may be delisted
ZIV
[*********************100%***********************]  1 of 1 completed

1 Failed download:
- ZIV: No data found for this date range, symbol may be delisted
ZIXI
[*********************100%***********************]  1 of 1 completed

1 Failed download:
- ZIXI: No data found, symbol may be delisted
ZLTQ
[*********************100%***********************]  1 of 1 completed

1 Failed download:
- ZLTQ: Period '5d' is invalid, must be one of ['1mo', '3mo', '6mo', 'ytd', '1y', '2y', '5y', '10y', 'max']
ZN
[*********************100%***********************]  1 of 1 completed

1 F

<ipython-input-56-aecd12de356d>:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  daily_hist['high_'+str(i)] = daily_hist['high'].shift(-1-i)
<ipython-input-56-aecd12de356d>:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  daily_hist['highest'] = daily_hist[["high_0", "high_1", "high_2", "high_3", "high_4"]].max(axis=1)
<ipython-input-56-aecd12de356d>:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See 

In [52]:
hist.empty

True

In [21]:
# print('Success rate is ' + str(final_table['success'].sum()/final_table['total'].sum()*100))

In [22]:
# final_table['success'].sum(), final_table['total'].sum()

In [23]:
# for i in np.unique(hist.index.date):
#     print(i)
#     sub_hist = hist[str(i):str(i)]

    
#     fig = plt.figure(figsize=(10, 7))

#     # Define position of 1st subplot
#     ax = fig.add_subplot(4, 1, 1)

#     # Set the title and axis labels
#     plt.title(symbol + ' Bollinger Bands')
#     plt.xlabel('Days')
#     plt.ylabel('Closing Prices')
#     plt.plot(sub_hist['close'], label='Closing Prices')
#     plt.plot(sub_hist['bollinger_up_14'], label='Bollinger Up')
#     plt.plot(sub_hist['bollinger_down_14'], label='Bollinger Down')
#     plt.legend()

#     # Define position of 2nd subplot
#     bx = fig.add_subplot(4, 1, 2)

#     # Set the title and axis labels
#     plt.title('Relative Strength Index')
#     plt.xlabel('Date')
#     plt.ylabel('RSI values')
    
#     plt.plot(sub_hist['MFI_14'], label='MFI_14')
#     plt.axhline(y=30, color='grey', linestyle='-')
#     plt.axhline(y=70, color='grey', linestyle='-')
#     plt.plot(sub_hist['RSI_14'], label='RSI_14') 

#     # Add a legend to the axis
#     plt.legend()

    
    
#     cx = fig.add_subplot(4, 1, 3)
#     plt.title('Pecentage Close Change')
#     plt.xlabel('Date')
#     plt.ylabel('Percentage values')
    
#     plt.plot(sub_hist['PER_close'], label='Percent close')

#     plt.legend()
    

#     cx = fig.add_subplot(4, 1, 4)
#     plt.title('Pecentage Volume Change')
#     plt.xlabel('Date')
#     plt.ylabel('Percentage values')
    
#     plt.plot(sub_hist['PER_volume'], label='Percent close')

#     plt.legend()
    
    
#     plt.tight_layout()
#     plt.show()
    
    
   

In [24]:
# for i in np.unique(hist.index.date):
#     print(i)
#     sub_hist = hist[str(i):str(i)]

#     for i in range(5):
#         sub_hist['high_'+str(i)] = sub_hist['high'].shift(-1-i)
#     sub_hist['highest'] = sub_hist[["high_0", "high_1", "high_2", "high_3", "high_4"]].max(axis=1)
#     sub_hist['success'] = np.where(sub_hist['close']*1.001 < sub_hist['highest'], 1,0) # 0.1%
#     print(sub_hist)

In [58]:
ttttable['expectation'].max()

1.491475850190992

In [64]:
list(set(ttttable.sort_values(['expectation']).tail(20)['ticket']))

['CLNE',
 'PACW',
 'LPSN',
 'ESPR',
 'GEVO',
 'SIRI',
 'MARA',
 'SGMO',
 'AGEN',
 'FCEL']

In [66]:
for i in list(set(ttttable.sort_values(['expectation']).tail(20)['ticket'])):
    print(ttttable[ttttable['ticket']==i])

     ticket  length  std  total  success    s_rate  expectation
2700   CLNE       7    1    411      351  0.854015     1.337490
2701   CLNE       7    2     22       19  0.863636     1.016118
2702   CLNE       7    3      0        0       NaN     1.000000
2703   CLNE      14    1    451      374  0.829268     1.345512
2704   CLNE      14    2     76       70  0.921053     1.066052
2705   CLNE      14    3      3        3  1.000000     1.003003
2706   CLNE      21    1    494      413  0.836032     1.393409
2707   CLNE      21    2     69       66  0.956522     1.064990
2708   CLNE      21    3      4        4  1.000000     1.004006
     ticket  length  std  total  success    s_rate  expectation
9306   PACW       7    1    427      373  0.873536     1.375458
9307   PACW       7    2     16       14  0.875000     1.012064
9308   PACW       7    3      0        0       NaN     1.000000
9309   PACW      14    1    449      389  0.866370     1.389266
9310   PACW      14    2     64       60